# Geometry-MMALS G1 - Grounded Functional Geometry

Colab-ready scaffold for the controlled RotatedMNIST experiment. It separates the frozen sensory geometry from context, route, host and synthesis geometry.

> **Status:** implementation scaffold, not a qualified G1 result. No quantum advantage is claimed.

**v1.0.1 audit patch:** this notebook is an executable scaffold for C0 and a declared G1-A supervised/grounded variant. It must not be read as evidence that C1-C5 have already passed.


## 0. Setup

The default clone URL assumes publication under `gharbonnier78/geometry-mmalls-g1`. In a local session, skip the clone and run `pip install -e .`.

In [ ]:
import os, pathlib
REPO_URL = 'https://github.com/gharbonnier78/geometry-mmalls-g1.git'
REPO_DIR = pathlib.Path('/content/geometry-mmalls-g1')
if pathlib.Path('/content').exists() and not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
if REPO_DIR.exists(): os.chdir(REPO_DIR)
!pip -q install -e .
print(pathlib.Path.cwd())

In [ ]:
from pathlib import Path
import copy, json, random, time
import numpy as np, pandas as pd, torch, yaml
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from geometry_mmalls.data import FixedAngleMNIST
from geometry_mmalls.geometry import normalized_stress, pairwise_fisher_rao, route_geodesic_loss
from geometry_mmalls.metrics import distance_order_correlation, euclidean_distance_matrix, neighborhood_preservation, pairwise_factor_distance
from geometry_mmalls.model import GeometryMMALS, SmallConvEncoder

config = yaml.safe_load(Path('configs/rotated_mnist_g1.yaml').read_text())
RUN_FULL, SEED = False, 0
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(DEVICE, config['data'])

## 1. Controlled data protocol

The true angle is retained for evaluation and declared geometry losses, but is never passed to the deployable router. Interpolation angles remain outside training and model selection.

In [ ]:
root = Path(config['data']['root'])
train_angles = list(map(float, config['data']['train_angles']))
interp_angles = list(map(float, config['data']['interpolation_angles']))
limit = None if RUN_FULL else 2000

def loader(angle, train, shuffle):
    ds = FixedAngleMNIST(root, angle=angle, train=train, download=True)
    if limit: ds = Subset(ds, range(min(limit, len(ds))))
    return DataLoader(ds, batch_size=128, shuffle=shuffle, num_workers=0)

train_loaders = {a: loader(a, True, True) for a in train_angles}
test_loaders = {a: loader(a, False, False) for a in train_angles}
interp_loaders = {a: loader(a, False, False) for a in interp_angles}

## 2. Pretrain and freeze the sensory grove

This boundary prevents geometry already learned by perception from being attributed automatically to MMALS.

In [ ]:
latent_dim = int(config['model']['latent_dim'])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
head = torch.nn.Linear(latent_dim, 10).to(DEVICE)
opt = torch.optim.AdamW(list(encoder.parameters()) + list(head.parameters()), lr=1e-3)
for epoch in range(1 if not RUN_FULL else int(config['training']['sensory_pretrain_epochs'])):
    encoder.train(); total = correct = 0
    for x, y, _, _ in train_loaders[0.0]:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = head(encoder(x)); loss = F.cross_entropy(logits, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        total += y.numel(); correct += (logits.argmax(1) == y).sum().item()
    print('sensory accuracy', correct/total)
sensory_state = copy.deepcopy(encoder.state_dict())

## 3. MMALS route-host model and visible G1 losses

In [ ]:
encoder.load_state_dict(sensory_state)
model = GeometryMMALS(
    encoder, latent_dim=latent_dim,
    context_dim=int(config['model']['context_dim']),
    num_hosts=int(config['model']['num_hosts']),
    host_hidden_dim=int(config['model']['host_hidden_dim']),
    freeze_encoder=True,
).to(DEVICE)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

def diversity(host_outputs):
    h = F.normalize(host_outputs, dim=-1)
    sim = torch.einsum('bhd,bjd->bhj', h, h)
    mask = ~torch.eye(sim.shape[-1], dtype=torch.bool, device=sim.device)
    return sim[:, mask].square().mean()

def train_stage(angle, epochs=1):
    model.train()
    for epoch in range(epochs):
        for x, y, u, _ in train_loaders[angle]:
            x, y, u = x.to(DEVICE), y.to(DEVICE), u.to(DEVICE)
            tr = model(x)
            ce = F.cross_entropy(tr.logits, y)
            # G1-A supervised/grounded variant: the true rotation angle u is used only
            # to shape the geometry loss during the controlled experiment. It is not
            # passed to the deployable router at inference time and must not be used
            # for claims outside the declared grounded-geometry protocol.
            geo = route_geodesic_loss(tr.route, u, bandwidth=20.0, margin=0.35)
            loss = ce + 0.1*geo + 0.05*diversity(tr.host_outputs)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return {'angle': angle, 'loss': float(loss.detach())}

## 4. Continual sequence

For qualification, add hardened EWC, replay, sparse-MoE, oracle-angle diagnostic and joint upper-bound comparisons with equal validation budgets.

In [ ]:
stage_logs, checkpoints = [], {}
for angle in train_angles:
    stage_logs.append(train_stage(angle, 1 if not RUN_FULL else int(config['training']['stage_epochs'])))
    checkpoints[angle] = copy.deepcopy(model.state_dict())
stage_logs

## 5. Trace collection and geometry evidence

In [ ]:
@torch.no_grad()
def collect(loaders, max_batches=3):
    model.eval(); rows=[]
    for angle, dl in loaders.items():
        for b, (x,y,u,idx) in enumerate(dl):
            if max_batches is not None and b >= max_batches: break
            tr=model(x.to(DEVICE))
            for i in range(len(y)):
                rows.append(dict(angle=float(u[i]), label=int(y[i]), prediction=int(tr.logits[i].argmax()),
                                 z0=tr.z0[i].cpu().numpy(), context=tr.context[i].cpu().numpy(),
                                 route=tr.route[i].cpu().numpy(), z_mm=tr.z_mm[i].cpu().numpy(),
                                 hosts=tr.host_outputs[i].cpu().numpy()))
    return pd.DataFrame(rows)
trace = collect({**test_loaders, **interp_loaders}, None if RUN_FULL else 3)
print(len(trace))

In [ ]:
stack=lambda name: np.stack(trace[name].to_numpy())
du=pairwise_factor_distance(trace.angle.to_numpy())
spaces={'sensory':euclidean_distance_matrix(stack('z0')),
        'context':euclidean_distance_matrix(stack('context')),
        'route':pairwise_fisher_rao(stack('route')),
        'synthesis':euclidean_distance_matrix(stack('z_mm'))}
rows=[]
for name,d in spaces.items():
    rho,p=distance_order_correlation(du,d)
    rows.append(dict(space=name,rho=rho,p_value=p,stress=normalized_stress(du,d),
                     knn=neighborhood_preservation(du,d,k=5),status='development'))
geometry_df=pd.DataFrame(rows); geometry_df

## 6. Host ablation evidence

A host is not specialized merely because it receives route mass. Measure the loss of true-class evidence when it is removed and the remaining route is renormalized.

In [ ]:
@torch.no_grad()
def ablations(loaders, max_batches=2):
    model.eval(); rows=[]
    for angle,dl in loaders.items():
        for b,(x,y,_,_) in enumerate(dl):
            if b>=max_batches: break
            x,y=x.to(DEVICE),y.to(DEVICE); tr=model(x)
            base=F.log_softmax(tr.logits,-1).gather(1,y[:,None]).squeeze(1)
            for h in range(tr.route.shape[1]):
                r=tr.route.clone(); r[:,h]=0; r=r/r.sum(1,keepdim=True).clamp_min(1e-8)
                z=torch.einsum('bh,bhd->bd',r,tr.host_outputs)
                out=model.classifier(model.synthesis_norm(z))
                impact=base-F.log_softmax(out,-1).gather(1,y[:,None]).squeeze(1)
                rows += [dict(angle=angle,host=h,ablation_impact=float(v),status='development') for v in impact.cpu()]
    return pd.DataFrame(rows)
ablation_df=ablations(test_loaders)
ablation_df.groupby(['angle','host']).ablation_impact.mean().unstack()

## 7. Minimal causal tangent probe

The qualified experiment must add matched-norm orthogonal controls, bootstrap confidence intervals and identity-preservation gates.

In [ ]:
trained=trace[trace.angle.isin(train_angles)]
Cmat=np.stack(trained.context); y=trained.angle.to_numpy(float)
tangent=np.linalg.lstsq(Cmat-Cmat.mean(0),y-y.mean(),rcond=None)[0]
tangent/=np.linalg.norm(tangent)+1e-12

@torch.no_grad()
def reroute(z0,c): return torch.softmax(model.router(torch.cat([z0,c],-1)),-1)
x=next(iter(interp_loaders[15.0]))[0].to(DEVICE); tr=model(x)
t=torch.tensor(tangent,dtype=tr.context.dtype,device=DEVICE)
pd.DataFrame([{'scale':s,'route_shift':float(torch.linalg.vector_norm(reroute(tr.z0,tr.context+s*t)-tr.route,dim=1).mean())}
              for s in [-2,-1,0,1,2]])

## 8. Export with epistemic status

In [ ]:
out=Path('results/per_seed/dev_seed_0'); out.mkdir(parents=True,exist_ok=True)
geometry_df.to_csv(out/'geometry_metrics.csv',index=False)
ablation_df.to_csv(out/'host_ablation.csv',index=False)
(out/'run_manifest.json').write_text(json.dumps({'status':'development_not_qualified','seed':SEED,
 'device':str(DEVICE),'timestamp':time.time(),'warning':'Not a qualified G1 result.'},indent=2))
print(out)

## 9. Before publication

Implement transport memory, dual memory tests, cross-seed host-role matching, compute accounting, statistical confidence intervals, baseline hardening and secondary-dataset replication. Follow `docs/CLAIMS_AND_GATES.md`.